# [A100 80GB] v4 2라운드 step 170 checkpoint · ViRL 검증 (사후 추가)

`ViRL_RoundGRPO_4B_A100_Train_v4.ipynb`의 2라운드는 step 175에서 vLLM/HF log-prob 검사로 멈췄습니다.
마지막 checkpoint(**step 170**, `round_02/train/checkpoint_00680_00170`)를 base·1라운드와 **같은 조건**(ViRL 296문항 × 4샘플, 같은 코드 묶음·설정·GPU)으로 검증합니다.
라운드를 더 돌릴 가치가 있는지 싸게 확인하는 용도입니다. 학습은 하지 않습니다.

**판정 규칙 (실행 전 고정, 2026-11-21):** 학습 노트북의 후보 선택(`choose()`)과 같은 기준입니다.
- **통과:** base 894 + 여유 12 = **906/1,184 이상**, 그리고 출력 길이 기준(잘림 비율 base +3%p 이하, 출력 중앙값 base의 70% 이상) 통과
  → MMMU 900(40GB) 평가는 1라운드 대신 **이 checkpoint**로 합니다.
- **미통과:** 라운드 2·3은 재시도하지 않습니다. 1라운드 MMMU 평가 여부는 따로 정합니다.
- 사전 지정 후보가 아니었으므로 보고서에 **사후 추가 검증**이라고 적습니다.

**실행:** 런타임 유형 **A100 GPU**, 학습과 같은 **80GB**가 잡혀야 합니다(다르면 4번 셀이 멈춤). 메뉴의 "모두 실행"으로 돌려도 됩니다.
설치 약 10분 + 이미지 풀기 약 5분 + 병합 약 5분 + 생성 약 15분(1,184샘플) ≈ **40~50분, 약 5~7유닛**.
(1차 실행은 이미지 풀기 단계가 없어 생성 시작 직전에 멈췄습니다. 그때 만들어진 요청·작업 파일은 그대로 재사용됩니다.)
4번 셀은 끝나거나 오류가 나면 60초 뒤 **런타임을 자동 종료**합니다(1~3번 셀 오류는 자동 종료 안 됨).
세션이 끊기면 처음부터 다시 실행하세요. 끝난 생성은 건너뜁니다.

**쓰는 파일:** `validation/round_02_step170/`(생성·채점·`posthoc_verdict.json`), `jobs/round_02_step170_*.json`, `logs/`, `stage_timing.jsonl`.
`selection_report.json`·`eval_handoff.json`·후보 목록은 **건드리지 않습니다.**


## 1. Drive 연결

In [1]:
import sys, json, os, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

RUN_TAG = 'round_grpo_v4'   # 학습 노트북과 같은 실행 폴더 (바꾸지 마세요)
WORK = Path('/content/virl_round_grpo')
ROOT = Path('/content/drive/MyDrive/VIRL39K_RFT_4B')
RUN_DIR = ROOT / RUN_TAG
WORK.mkdir(parents=True, exist_ok=True)
assert (RUN_DIR / 'config.json').is_file(), f'v4 실행 폴더가 없습니다: {RUN_DIR}'
print('실행 폴더:', RUN_DIR)

Mounted at /content/drive
실행 폴더: /content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4


## 2. v4 코드 묶음 (학습 노트북과 같은 파일, 해시 검사)

In [2]:
import base64, hashlib, io, zipfile
BUNDLE_B64 = '...'
BUNDLE_SHA256 = '6caeb94cae4fa7d58b370fe678fc159b0f6462bd5acc80a6ab33e2ea5e36fda9'
payload = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(payload).hexdigest() == BUNDLE_SHA256
BUNDLE_DIR = WORK / ('code_' + BUNDLE_SHA256[:12])
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        dest = (BUNDLE_DIR / name).resolve()
        assert dest.is_relative_to(BUNDLE_DIR.resolve())
        data = z.read(name)
        if dest.exists():
            assert dest.read_bytes() == data, f'코드가 바뀌었습니다: {dest}'
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(data)
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
from common import atomic_json, read_json, immutable_json
from orchestrate import run
print('코드:', BUNDLE_DIR)


코드: /content/virl_round_grpo/code_6caeb94cae4f


## 3. 실행 환경 설치 (학습 노트북과 같음)

In [3]:
run(sys.executable, 'setup_env.py', ['--work', WORK], WORK / 'installation.log')
TRAIN_PY = str(WORK / 'train_env/bin/python')
GEN_PY = str(WORK / 'gen_env/bin/python')
run(TRAIN_PY, 'test_suite.py', [], WORK / 'cpu_tests.log')
print('설치 및 CPU/Torch 손실 검사 통과')


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: virl_round_grpo/train_env
Activate with: source virl_round_grpo/train_env/bin/activate
Using Python 3.12.3 environment at: virl_round_grpo/train_env
Resolved 71 packages in 413ms
Prepared 71 packages in 30.17s
Installed 71 packages in 220ms
 + accelerate==1.11.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + antlr4-python3-runtime==4.13.2
 + anyio==4.16.0
 + attrs==26.1.0
 + av==19.0.2
 + certifi==2026.11.20
 + charset-normalizer==3.5.2
 + datasets==4.3.0
 + dill==0.4.0
 + filelock==4.2.12
 + frozenlist==1.8.0
 + fsspec==2025.9.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==0.36.0
 + idna==3.20
 + jinja2==3.1.6
 + latex2sympy2-extended==1.10.2
 + markupsafe==3.0.4
 + math-verify==0.8.0
 + mpmath==1.3.0
 + multidict==7.2.1
 + multiprocess==0.70.16
 + networkx==3.7
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu

## 4. step 170 checkpoint 검증과 판정

이 셀이 도는 동안 다른 셀을 실행하지 마세요.

In [4]:
import math
from orchestrate import Workflow
from common import policy_id, read_rows, timestamp

CAND_NAME = 'round_02_step170'
ADAPTER = RUN_DIR / 'round_02/train/checkpoint_00680_00170'
EXPECTED_POLICY_ID = '55922b8dfc2948d12f6e5190f02be0dd92804aa2ba0b78a19d2a3d533bcfc442'
AUTO_SHUTDOWN = True   # 끝나거나 오류가 나면 60초 뒤 런타임 종료


def posthoc_verdict(base, cand, cfg, margin):
    # orchestrate.Workflow.choose()와 같은 기준
    why = []
    if cand['truncated_fraction'] > base['truncated_fraction'] + cfg['max_val_truncation_increase']:
        why.append('truncation_increase')
    if cand['output_tokens_median'] < cfg['min_val_median_length_ratio'] * base['output_tokens_median']:
        why.append('median_length_drop')
    return (not why and cand['correct'] >= base['correct'] + margin), why


def paired(ref_rows, cand_rows):
    def per_q(rows):
        out = {}
        for r in rows:
            out[r['uid']] = out.get(r['uid'], 0) + int(r['reward'])
        return out
    a, b = per_q(ref_rows), per_q(cand_rows)
    assert set(a) == set(b), '비교할 문항 목록이 다릅니다'
    better = sum(b[u] > a[u] for u in a)
    worse = sum(b[u] < a[u] for u in a)
    n, k = better + worse, min(better, worse)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    return {'better': better, 'worse': worse, 'same': len(a) - n, 'sign_test_p_two_sided': p,
            'net_samples': sum(b.values()) - sum(a.values())}


def show(name, r):
    print(f"  {name:<18} {r['correct']:>4}/{r['n']} ({r['accuracy']:.2%})  "
          f"잘림 {r['truncated']} ({r['truncated_fraction']:.2%})  출력 중앙 {r['output_tokens_median']}")


try:
    import torch
    CONFIG_FILE = RUN_DIR / 'config.json'
    cfg = read_json(CONFIG_FILE)
    assert cfg['run_dir'] == str(RUN_DIR) and cfg['work_dir'] == str(WORK), '실행/작업 폴더가 학습 때와 다릅니다'
    assert cfg['train_python'] == TRAIN_PY and cfg['gen_python'] == GEN_PY, '설치 경로가 학습 때와 다릅니다'
    latest = read_json(RUN_DIR / 'round_02/train/latest.json')
    assert (latest['adapter'] == str(ADAPTER) and latest['optimizer_step'] == 170
            and latest['policy_id'] == EXPECTED_POLICY_ID), f'latest.json이 예상과 다릅니다: {latest}'
    assert policy_id(ADAPTER) == EXPECTED_POLICY_ID, 'checkpoint 파일이 학습 때와 다릅니다(해시 불일치)'
    assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
    hw = read_json(RUN_DIR / 'training_hardware.json')
    gpu = torch.cuda.get_device_name(0)
    assert gpu == hw['gpu'], (f'base 검증과 같은 GPU에서만 비교합니다. 기록: {hw["gpu"]} / 현재: {gpu}\n'
                              '→ 런타임 > 런타임 연결 해제 및 삭제 후 다시 연결하세요.')
    base = read_json(RUN_DIR / 'validation/base/report.json')
    assert base['correct'] == read_json(RUN_DIR / 'selection_report.json')['baseline']['correct']
    r1 = read_json(RUN_DIR / 'candidates/round_01.json')
    assert r1['name'] == 'round_01' and r1['report']['n'] == base['n']
    margin = max(1, math.ceil(cfg['min_val_gain'] * base['n']))
    print('GPU:', gpu, '| checkpoint 해시 확인:', EXPECTED_POLICY_ID[:16])
    print(f"통과 기준: {base['correct'] + margin}/{base['n']} 이상 (base {base['correct']} + 여유 {margin}) + 출력 길이 기준")

    flow = Workflow(CONFIG_FILE)
    # 새 런타임에는 학습 이미지가 없다: v4와 같은 prepare 단계로 Drive pool zip을 Colab 로컬에 풀고 고정 데이터·이미지 해시 검사
    flow.call('train', 'prepare.py', ['--config', CONFIG_FILE], 'prepare')
    report = flow.validate(str(ADAPTER), CAND_NAME)
    assert report['n'] == base['n']

    passed, why = posthoc_verdict(base, report, cfg, margin)
    cand_rows = read_rows(RUN_DIR / 'validation' / CAND_NAME / 'scores.jsonl')
    vs_base = paired(read_rows(RUN_DIR / 'validation/base/scores.jsonl'), cand_rows)
    vs_r1 = paired(read_rows(RUN_DIR / 'validation/round_01/scores.jsonl'), cand_rows)
    decision = ('통과: MMMU 900(40GB) 평가는 round_01 대신 이 checkpoint로 (사후 추가 후보, 보고서에 명시)' if passed else
                '미통과: 라운드 2·3 재시도 안 함. round_01 MMMU 평가 여부는 따로 결정')
    result = {'candidate': CAND_NAME, 'adapter': str(ADAPTER), 'policy_id': EXPECTED_POLICY_ID, 'gpu': gpu,
              'report': report, 'baseline': base, 'round_01': r1['report'], 'margin_samples': margin,
              'threshold_correct': base['correct'] + margin, 'passed': passed, 'length_guard_reasons': why,
              'paired_vs_base': vs_base, 'paired_vs_round_01': vs_r1, 'decision': decision,
              'rule': 'post-hoc (2026-11-21): same as choose(); >= base+margin and length guards -> MMMU eval this checkpoint instead of round_01',
              'created': timestamp()}
    atomic_json(RUN_DIR / 'validation' / CAND_NAME / 'posthoc_verdict.json', result)

    print('\nViRL 검증 (296문항 × 4샘플)')
    show('base', base)
    show('round_01', r1['report'])
    show(CAND_NAME, report)
    print(f"\n문항별 vs base: 나아짐 {vs_base['better']} / 나빠짐 {vs_base['worse']} / 같음 {vs_base['same']} "
          f"(sign test p={vs_base['sign_test_p_two_sided']:.3f})")
    print(f"문항별 vs round_01: 나아짐 {vs_r1['better']} / 나빠짐 {vs_r1['worse']} / 같음 {vs_r1['same']} "
          f"(sign test p={vs_r1['sign_test_p_two_sided']:.3f})")
    if why:
        print('출력 길이 기준 탈락:', why)
    print('\n판정:', decision)
    print('기록:', RUN_DIR / 'validation' / CAND_NAME / 'posthoc_verdict.json')
finally:
    if AUTO_SHUTDOWN:
        from google.colab import runtime
        print('끝(또는 오류). 60초 뒤 런타임을 종료합니다.')
        import time; time.sleep(60)
        runtime.unassign()

GPU: NVIDIA A100-SXM4-80GB | checkpoint 해시 확인: 55922b8dfc2948d1
통과 기준: 906/1184 이상 (base 894 + 여유 12) + 출력 길이 기준
Reusing frozen train/validation data: 1160 296

Fetching 2 files: 100%|██████████| 2/2 [00:21<00:00, 10.87s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 21.70it/s]
Merged: /content/virl_round_grpo/merged/55922b8dfc2948d12f6e
INFO 11-21 05:16:29 [__init__.py:216] Automatically detected platform cuda.
INFO 11-21 05:16:37 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'seed': 3407, 'max_model_len': 14336, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 1}, 'generation_config': 'vllm', 'model': '/content/virl_round_grpo/merged/55922b8dfc2948d12f6e'}
INFO 11-21 05:16:49 [model.py:547] Resolved architecture: Qwen3VLForConditionalGeneration
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 11-21 05:16:49 [model.py:1510] Using max model len 14336
INFO 11-21 05:16:50 [scheduler.py:205] Chunked prefill is e